In [ ]:
MERGED_REPO = "michi883/memory-moment-v5-gemma4-e2b-merged"
LORA_REPO = "michi883/memory-moment-v5-gemma4-e2b-lora"
LOCAL_DIR = "/content/merged_model"
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
%%capture
!pip install unsloth
# Latest Unsloth nightly has the freshest Gemma 4 fixes
!pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = "/content/memory_moment_train_v5.jsonl",
    split = "train",
)

# Apply Gemma 4's chat template to the messages field
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize = False,
            add_generation_prompt = False,
        )
    }

dataset = dataset.map(format_chat)
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

<bos><|turn>system
You help a family caregiver find words that work in a hard moment with someone they love who has memory loss. The caregiver provides context about the loved one — their relationship, voice, things they often say, what calms them, what to avoid, what they resist — followed by a situation. Produce 3 to 5 short lines the caregiver can say out loud, each tagged with a mode: 🤝 agree-and-move, 🙂 lighten, 🫱 redirect, 🛑 firm-but-kind, or 🎯 distract. Use phrases from "often says" when they fit naturally, or paraphrase the loved one's voice when they don't. Skip modes that don't fit. Output only the lines.<turn|>
<|turn>user
Your husband. You call him Art.
Voice: sly, dry.
He often says: "Well well." / "You first." / "Is that so."
He calms with a walk, mornings are harder.

Situation: He's making the same joke about the toaster for the third time this morning.<turn|>
<|turn>model
🤝 You said that one already. I heard it.
🙂 Third time, Art. You like that joke.
🫱 Tell me again at

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 12,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/32 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32 | Num Epochs = 12 | Total steps = 48
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Step,Training Loss
5,5.367422
10,2.637696
15,1.337102
20,0.673924
25,0.430569
30,0.314176
35,0.250744
40,0.199267
45,0.165833


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-48/tokenizer_config.json.


In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You help a family caregiver find words that work in a hard moment "
    "with someone they love who has memory loss. Given a phrase the person "
    "would say, a situation, and context about them, produce short lines "
    "the caregiver can say out loud. Each line should feel like something "
    "she would accept. Tag each line with a mode: 🤝 agree-and-move, "
    "🙂 lighten, 🫱 redirect, 🛑 firm-but-kind, or 🎯 distract. Produce 3 to 5 "
    "lines. Skip modes that don't fit. Output only the lines, one per line, "
    "mode emoji first."
)

# Pick a test case — ideally one NOT in the training set
user_prompt = (
    'Phrase: "He means well"\n'
    'Situation: She is annoyed her husband forgot to take the bins out again.\n'
    'Context: Husband is Frank. She calms down with tea.'
)

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT}],
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": user_prompt}],
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    attention_mask=torch.ones_like(inputs),  # fixes the attention_mask warning
    max_new_tokens=200,                       # bumped — 50 was too short for 5 lines
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

🤝 I know you think I forgot.
🙂 Happens to the best of us.
🫱 Come sit. We will do it together.
🛑 We need to get these out now.
🎯 Open the bin for me.


In [ ]:
from huggingface_hub import HfApi, create_repo
import os

print(f"Uploading LoRA adapter to {LORA_REPO}...")
model.push_to_hub(LORA_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(LORA_REPO, token=HF_TOKEN)

print("Merging adapter...")
merged_model = model.merge_and_unload()

print(f"Saving to {LOCAL_DIR}...")
os.makedirs(LOCAL_DIR, exist_ok=True)
merged_model.save_pretrained(LOCAL_DIR, safe_serialization=True)
tokenizer.save_pretrained(LOCAL_DIR)

print(f"Uploading to {MERGED_REPO}...")
create_repo(MERGED_REPO, token=HF_TOKEN, exist_ok=True, private=False)
HfApi(token=HF_TOKEN).upload_folder(
    folder_path=LOCAL_DIR,
    repo_id=MERGED_REPO,
    repo_type="model",
    commit_message=f"Upload merged model",
)

print("Done.")

Uploading LoRA adapter to michi883/memory-moment-v5-gemma4-e2b-lora...


README.md:   0%|          | 0.00/536 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|3         | 4.73MB /  124MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...rged_model/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            

  ...d_model/model.safetensors:   0%|          | 24.0MB / 10.2GB            

Done.
